# PlanForge
> Your AI business advisor. Tell it what you want to achieve, have a conversation, get a plan.

*v0.5.0*

In [ ]:
# @title Setup
# @markdown Run this cell every session.
# @markdown
# @markdown First time? [Getting started guide](https://planforge.pysolvr.com/docs/getting-started)
# @markdown - Save a copy: File > Save a copy in Drive
# @markdown - Add API key: click key icon (left sidebar) > add `PLANFORGE_API_KEY`
!pip install -q -U pysolvr-client
import sys
from google.colab import userdata, drive

try:
    API_KEY = userdata.get('PLANFORGE_API_KEY')
except userdata.SecretNotFoundError:
    from IPython.display import HTML, display
    display(HTML('<div style="background:#1e293b;border:1px solid #ef4444;border-radius:8px;padding:16px;font-family:Inter,system-ui,sans-serif;color:#f1f5f9"><b style="color:#ef4444">API key not found in Colab Secrets</b><ol style="color:#94a3b8;font-size:13px;margin-top:8px"><li>Click the key icon in the left sidebar</li><li>Add secret: <code>PLANFORGE_API_KEY</code></li><li>Paste your API key as the value</li><li>Toggle Notebook access ON</li><li>Re-run this cell</li></ol></div>'))
    assert False, 'API key not configured - see instructions above'

drive.mount('/content/drive', force_remount=False)
sys.path.insert(0, '/content')

from pysolvr_client import ApiClient, Display, DriveManager

API_BASE = 'https://ynmwpf8duf.execute-api.us-east-1.amazonaws.com/dev/'
client = ApiClient(API_BASE, API_KEY)
ui = Display('#3B82F6', '#10B981')
drive_mgr = DriveManager('planforge', ['profiles', 'plans', 'exports', 'uploads'])
drive_mgr.ensure_folders()

if client.health_check():
    ui.success('Connected to PlanForge API', f'Drive: {drive_mgr.root}')
else:
    ui.error('Could not connect to API', 'Check your API key and try again')
import ipywidgets as widgets
import time as _time
from IPython.display import clear_output
_state = {'goal': None, 'goal_info': {}, 'topics': [], 'current_topic': None, 'history': {}, 'locked': {}}


In [ ]:
# @title What do you want to achieve?
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

_goal_output = widgets.Output()
_PRIMARY = '#3B82F6'

_CATEGORY_LABELS = {
    'Grants': 'I need a grant',
    'Loans': 'I need a loan',
    'Investment': 'I want to raise investment',
    'Visas': 'I need a visa for my business',
    'Accelerators': "I'm applying to an accelerator",
    'Validation': 'I want to test my idea',
}
_CAT_ORDER = ['Grants', 'Loans', 'Investment', 'Visas', 'Accelerators', 'Validation']

def _pill_btn(label, on_click, width='auto'):
    b = widgets.Button(description=label, layout=widgets.Layout(margin='4px', width=width, height='36px'))
    b.style.button_color = '#1e293b'
    b.style.text_color = 'white'
    b.on_click(on_click)
    return b

def _select_goal():
    result = client.call('GET', '/goals')
    if not result['ok']:
        with _goal_output:
            clear_output(wait=True)
            ui.error('Could not load goals', result.get('error', 'Unknown error'))
        return

    goals = result['data']['goals']
    categories = {}
    for g in goals:
        cat = g.get('category', 'Other')
        categories.setdefault(cat, []).append(g)

    sorted_cats = [c for c in _CAT_ORDER if c in categories]
    sorted_cats += [c for c in categories if c not in _CAT_ORDER]

    def _render_categories():
        with _goal_output:
            clear_output(wait=True)
            ui.card('What do you want to achieve?', 'Pick the option that best describes your goal.')
            box = widgets.VBox(layout=widgets.Layout(margin='8px 0'))
            btns = []
            for cat in sorted_cats:
                label = _CATEGORY_LABELS.get(cat, cat)
                btns.append(_pill_btn(label, lambda _, c=cat: _render_goals(c)))
            box.children = btns
            display(box)

    def _render_goals(category):
        cat_goals = categories[category]
        # If only one goal in category and it has targets, skip straight to targets
        if len(cat_goals) == 1 and cat_goals[0].get('targets'):
            _render_targets(cat_goals[0], category)
            return
        # If only one goal and no targets, confirm directly
        if len(cat_goals) == 1:
            _confirm_goal(cat_goals[0])
            return
        with _goal_output:
            clear_output(wait=True)
            label = _CATEGORY_LABELS.get(category, category)
            ui.card(label, 'Which type?')
            back = _pill_btn('< Back', lambda _: _render_categories(), width='80px')
            display(back)
            box = widgets.VBox(layout=widgets.Layout(margin='8px 0'))
            btns = []
            for g in cat_goals:
                b = _pill_btn(g['name'], lambda _, goal=g: _on_goal_click(goal, category))
                b.tooltip = g.get('description', '')
                btns.append(b)
            box.children = btns
            display(box)

    def _on_goal_click(goal, category):
        if goal.get('targets'):
            _render_targets(goal, category)
        else:
            _confirm_goal(goal)

    def _render_targets(goal, category):
        with _goal_output:
            clear_output(wait=True)
            ui.card(goal['name'], 'Who are you applying to? Pick the closest match for tailored guidance.')
            back = _pill_btn('< Back', lambda _: _render_goals(category), width='80px')
            display(back)
            box = widgets.VBox(layout=widgets.Layout(margin='8px 0'))
            btns = []
            for t in goal['targets']:
                b = _pill_btn(t['name'], lambda _, target=t, g=goal: _confirm_goal(g, target))
                b.tooltip = t.get('description', '')
                btns.append(b)
            box.children = btns
            display(box)

    def _confirm_goal(goal, target=None):
        _state['goal'] = goal['id']
        _state['goal_info'] = goal
        _state['target'] = target['id'] if target and target['id'] != 'general' else None
        _state['topics'] = [{'id': t['id'], 'name': t['name']} for t in goal.get('required_topics', [])]
        _state['current_topic'] = _state['topics'][0]['id'] if _state['topics'] else None
        _state['history'] = {}
        _state['locked'] = {}
        names = [t['name'] for t in _state['topics']]
        target_note = f' <b>Target:</b> {target["name"]}<br>' if target and target['id'] != 'general' else ''
        with _goal_output:
            clear_output(wait=True)
            ui.card(
                f'Goal: {goal["name"]}',
                f'<b>Audience:</b> {goal.get("audience", "you")}<br>'
                f'{target_note}'
                f'<b>Topics to cover:</b> {", ".join(names)}<br><br>'
                f'Run the next cell to start your discovery conversation.',
                status='success'
            )

    _render_categories()

display(_goal_output)
_select_goal()


In [ ]:
# @title Discovery chat

if not _state.get('goal') or not _state.get('topics'):
    ui.card('No goal selected', 'Run the <b>goal selection</b> cell above first.', status='error')
    assert False, 'Goal not set'

if not isinstance(_state.get('history'), dict):
    _state['history'] = {}

# --- Widgets ---
_counter_out = widgets.Output()
_chat_out = widgets.Output(layout={'border': '1px solid #334155', 'border_radius': '12px',
                                    'padding': '16px', 'max_height': '400px',
                                    'overflow_y': 'auto'})
_input_area = widgets.Textarea(placeholder='Type your message or paste text (CV, financials, notes)...',
                               layout=widgets.Layout(width='100%', height='80px'))
_send_btn = widgets.Button(description='Send', button_style='primary',
                            layout=widgets.Layout(width='80px', height='34px'))
_clear_btn = widgets.Button(description='Clear', button_style='warning',
                             layout=widgets.Layout(width='80px', height='34px'))
_status_out = widgets.Output()
_completion_out = widgets.Output()
_lock_pending = [False]

# Topic pill buttons
_pill_btns = []
for _t in _state['topics']:
    _b = widgets.Button(description=_t['name'],
                        layout=widgets.Layout(height='26px', margin='2px', padding='0 8px',
                                             border_radius='12px'))
    _b._topic_id = _t['id']
    _pill_btns.append(_b)


def _all_locked():
    return all(t['id'] in _state['locked'] for t in _state['topics'])


def _get_history():
    topic = _state['current_topic'] or 'default'
    if topic not in _state['history']:
        _state['history'][topic] = []
    return _state['history'][topic]


def _show_completion():
    with _completion_out:
        clear_output(wait=True)
        if _all_locked():
            _input_area.layout.display = 'none'
            _send_btn.layout.display = 'none'
            _clear_btn.layout.display = 'none'
            ui.card('Discovery complete',
                    'All topics locked. Run the <b>next cell</b> to generate your plan.',
                    status='success')
        else:
            _input_area.layout.display = ''
            _send_btn.layout.display = ''
            _clear_btn.layout.display = ''


def _render_progress():
    with _counter_out:
        clear_output(wait=True)
        done = sum(1 for t in _state['topics'] if t['id'] in _state['locked'])
        total = len(_state['topics'])
        pct = int(done / total * 100) if total else 0
        html = (f'<div style="font-family:Inter,sans-serif">'
                f'<span style="font-size:12px;color:#94a3b8">{done}/{total} topics</span>'
                f'<div style="background:#334155;border-radius:4px;height:6px;margin:4px 0 6px 0">'
                f'<div style="background:#10b981;height:6px;border-radius:4px;width:{pct}%"></div></div>'
                f'</div>')
        display(HTML(html))
    for b in _pill_btns:
        tid = b._topic_id
        name = next(t['name'] for t in _state['topics'] if t['id'] == tid)
        if tid in _state['locked']:
            b.button_style = 'success'
            b.description = '\u2713 ' + name
        elif tid == _state['current_topic']:
            b.button_style = 'primary'
            b.description = '\u25cf ' + name
        else:
            b.button_style = ''
            b.description = name
    _show_completion()


def _render_chat(show_lock=False):
    with _chat_out:
        clear_output(wait=True)
        msgs = _get_history()
        if not msgs:
            display(HTML(''))
            return
        html = ''
        for msg in msgs:
            if msg['role'] == 'user':
                text = msg['content'][:500] + ('...' if len(msg['content']) > 500 else '')
                html += (f'<div style="text-align:right;margin:6px 0">'
                         f'<span style="background:#1e40af;color:#f1f5f9;padding:8px 12px;'
                         f'border-radius:14px 14px 4px 14px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{text}</span></div>')
            else:
                html += (f'<div style="text-align:left;margin:6px 0">'
                         f'<span style="background:#1e293b;color:#e2e8f0;padding:8px 12px;'
                         f'border-radius:14px 14px 14px 4px;display:inline-block;max-width:80%;'
                         f'text-align:left;font-size:13px;font-family:Inter,sans-serif;'
                         f'white-space:pre-wrap">{msg["content"]}</span></div>')
        display(HTML(html))
        if show_lock:
            _inline_lock = widgets.Button(description='\u2713 Lock in', button_style='success',
                                          layout=widgets.Layout(width='100px', height='30px', margin='8px 0 0 0'))
            _inline_lock.on_click(_do_lock)
            display(_inline_lock)


def _switch_topic(btn):
    topic_id = btn._topic_id
    if topic_id == _state['current_topic']:
        return
    _state['current_topic'] = topic_id
    _lock_pending[0] = False
    _render_progress()
    _render_chat()
    with _status_out:
        clear_output(wait=True)


for _b in _pill_btns:
    _b.on_click(_switch_topic)


def _on_clear(btn):
    topic = _state['current_topic'] or 'default'
    _state['history'][topic] = []
    _lock_pending[0] = False
    _render_chat()
    with _status_out:
        clear_output(wait=True)
        display(HTML('<span style="color:#94a3b8;font-size:12px">Chat cleared</span>'))


def _on_send(btn):
    if _all_locked():
        return
    message = _input_area.value.strip()
    if not message:
        with _status_out:
            clear_output(wait=True)
            display(HTML('<span style="color:#f87171;font-size:12px">Type a message</span>'))
        return
    msgs = _get_history()
    msgs.append({'role': 'user', 'content': message})
    _input_area.value = ''
    _lock_pending[0] = False
    _render_chat()
    with _status_out:
        clear_output(wait=True)
        display(HTML('<span style="color:#94a3b8;font-size:12px">Thinking...</span>'))
    topic = _state['current_topic'] or 'business_overview'
    body = {'topic': topic, 'message': message, 'goal_type': _state['goal'],
            'conversation_history': [{'role': m['role'], 'content': m['content'][:1000]}
                                     for m in msgs[-6:]],
            'locked_topics': _state['locked']}
    submit = client.call('POST', '/chat', body)
    if not submit['ok']:
        msgs.append({'role': 'assistant', 'content': f"Error: {submit.get('error', 'Failed')}"})
        with _status_out:
            clear_output(wait=True)
        _render_chat()
        return
    job_id = submit['data'].get('job_id')
    if not job_id:
        reply = submit['data'].get('response', '')
        msgs.append({'role': 'assistant', 'content': reply})
        with _status_out:
            clear_output(wait=True)
        _render_chat()
        return
    for _ in range(60):
        _time.sleep(2)
        poll = client.call('GET', f'/jobs/{job_id}')
        if not poll['ok']:
            break
        if poll['data'].get('status') == 'complete':
            rd = poll['data'].get('result', {})
            msgs.append({'role': 'assistant', 'content': rd.get('response', '')})
            show_lock = rd.get('topic_status') in ('suggested_lock', 'locked')
            if rd.get('topic_status') == 'locked':
                _do_lock()
            else:
                _lock_pending[0] = show_lock
                with _status_out:
                    clear_output(wait=True)
                _render_chat(show_lock=show_lock)
            return
        elif poll['data'].get('status') == 'failed':
            msgs.append({'role': 'assistant', 'content': f"Error: {poll['data'].get('error', 'Failed')}"})
            break
    with _status_out:
        clear_output(wait=True)
    _render_chat()


def _do_lock(btn=None):
    topic = _state['current_topic']
    if not topic:
        return
    client.call('POST', '/chat', {'topic': topic, 'message': 'lock_in', 'goal_type': _state['goal'],
                                   'conversation_history': [], 'locked_topics': _state['locked']})
    # Store conversation content (not just 'locked') so generate has real data
    msgs = _state['history'].get(topic, [])
    summary = '\n'.join(m['content'] for m in msgs if m['role'] == 'user')
    _state['locked'][topic] = summary or 'No details provided'
    _lock_pending[0] = False
    # Move to next unlocked topic
    for t in _state['topics']:
        if t['id'] not in _state['locked']:
            _state['current_topic'] = t['id']
            with _status_out:
                clear_output(wait=True)
                display(HTML(f'<span style="color:#10b981;font-size:12px">Locked! Next: {t["name"]}</span>'))
            break
    _render_progress()
    _render_chat()


_send_btn.on_click(_on_send)
_clear_btn.on_click(_on_clear)
_render_progress()
_render_chat()
display(widgets.VBox([
    _counter_out,
    widgets.HBox(_pill_btns, layout=widgets.Layout(flex_flow='row wrap', margin='0 0 8px 0')),
    _chat_out,
    widgets.HBox([_input_area, widgets.VBox([_send_btn, _clear_btn])]),
    _status_out,
    _completion_out,
]))


In [ ]:
# @title Generate my plan
result = client.run_async('/generate', {'goal_type': _state['goal'],
             'locked_topics': _state.get('locked', {}),
             'target': _state.get('target')})
if result['ok']:
    data = result['data']
    plan_id = data.get('plan_id', '')
    content = data.get('content', data.get('plan', ''))
    ui.card('Your Plan', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{content}</pre>', status='success')
    if plan_id:
        filename = f'{_state["goal"]}_{plan_id[:8]}.md'
        drive_mgr.save('plans', filename, content, meta={'plan_id': plan_id, 'goal': _state['goal']})
        ui.success('Saved to Drive', f'plans/{filename}')
else:
    ui.error(result.get('error', 'Generation failed'), f'Status: {result.get("status")}')


In [ ]:
# @title Export to a different format
FORMAT = 'executive_summary'  # @param ["executive_summary", "pitch_deck", "one_pager", "financial_summary", "full_pdf"]

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    result = client.run_async('/export', {'content': content, 'format': FORMAT})
    if result['ok']:
        data = result['data']
        ui.card(f'Export: {FORMAT}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
        filename = f'{FORMAT}_{plan_files[-1].replace(".md", "")}.md'
        drive_mgr.save('exports', filename, data['content'], meta={'format': FORMAT})
        ui.success('Saved', f'exports/{filename}')
    else:
        ui.error(result.get('error', 'Export failed'))


In [ ]:
# @title Refine a section
SECTION = ''  # @param {type:"string"}
FEEDBACK = ''  # @param {type:"string"}

plan_files = drive_mgr.list_files('plans')
if not plan_files:
    ui.error('No plans found', 'Generate a plan first')
elif not SECTION.strip():
    ui.error('Enter a section name', 'e.g. Market Analysis, Financial Projections')
else:
    content = drive_mgr.load('plans', plan_files[-1])
    body = {'content': content, 'section': SECTION}
    if FEEDBACK.strip():
        body['feedback'] = FEEDBACK
    result = client.run_async('/refine', body)
    if result['ok']:
        data = result['data']
        ui.card(f'Refined: {data["section"]}', f'<pre style="white-space:pre-wrap;color:#f1f5f9">{data["content"]}</pre>', status='success')
    else:
        ui.error(result.get('error', 'Refinement failed'))


In [ ]:
# @title My plans
result = client.call('GET', '/plans')
if result['ok']:
    plans = result['data'].get('plans', [])
    if not plans:
        ui.card('Plans', 'No plans generated yet.')
    else:
        rows = [{'ID': p['plan_id'][:8], 'Goal': p.get('goal_type', ''), 'Created': p.get('created_at', '')[:10]} for p in plans]
        ui.table(rows)
else:
    ui.error(result.get('error', 'Could not fetch plans'))


In [ ]:
# @title Account
import requests as _req
result_acct = client.call('GET', '/account')
result_usage = client.get_usage()

sub_html = ''
if result_acct['ok']:
    d = result_acct['data']
    sub_html = (f"<b>{d.get('plan', 'Free')}</b><br>"
        f"Status: {d.get('status', 'active')}<br>"
        f"<a href='https://planforge.pysolvr.com/pricing' style='color:#60a5fa'>Manage plan</a>")
else:
    sub_html = f"<span style='color:#f87171'>{result_acct.get('error', 'Could not fetch account')}</span>"

usage_html = ''
if result_usage['ok']:
    data = result_usage['data']
    usage_html = ui.usage_bar_html(data.get('current_month_spend_usd', 0), data.get('monthly_limit_usd', 1))
else:
    usage_html = f"<span style='color:#f87171'>{result_usage.get('error', 'Could not fetch usage')}</span>"

support_html = ('<p><b>Files:</b> Google Drive > pysolvr > planforge</p>'
    '<p><b>Docs:</b> <a href="https://planforge.pysolvr.com/docs" style="color:#60a5fa">planforge.pysolvr.com/docs</a></p>'
    '<p><b>Support:</b> <a href="mailto:support@pysolvr.com?subject=PlanForge" style="color:#60a5fa">support@pysolvr.com</a></p>')

_current_ver = '0.5.0'
_latest_ver = _current_ver
try:
    _cl = _req.get('https://raw.githubusercontent.com/dAIvdMercer/planforge-client/main/CHANGELOG.md', timeout=5).text
    import re as _re
    _m = _re.search(r'## (\d+\.\d+\.\d+)', _cl)
    if _m: _latest_ver = _m.group(1)
except: pass

if _current_ver == _latest_ver:
    _ver_status = '<span style="color:#10b981">Up to date</span>'
else:
    _ver_status = f'<span style="color:#f59e0b">Update available: v{_latest_ver}</span><br><small>Save a fresh copy from the link below</small>'
version_html = (f'<p><b>Current:</b> v{_current_ver}</p>'
    f'<p><b>Latest:</b> v{_latest_ver} — {_ver_status}</p>'
    '<p><a href="https://github.com/dAIvdMercer/planforge-client/blob/main/CHANGELOG.md" style="color:#60a5fa">View changelog</a></p>')

ui.tabs({'Subscription': sub_html, 'Usage': usage_html, 'Version': version_html, 'Support': support_html})
